In [ ]:
# 1. Install system dependencies (GPU detection & utilities)
!apt-get update -qq
!apt-get install -y pciutils lshw zstd curl

# 2. Install Ollama and Python helpers
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q huggingface_hub pyngrok

print("✅ Setup Phase 1 Complete: Dependencies installed.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
lshw is already the newest version (02.19.git.2021.06.19.996aaad9c7-2build1).
pciutils is already the newest version (1:3.7.0-6).
zstd is already the newest version (1.4.8+dfsg-3build1).
curl is already the newest version (7.81.0-1ubuntu1.22).
0 upgraded, 0 newly installed, 0 to remove and 94 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.

In [ ]:
import os
import subprocess
import time

# 1. PREPEND the paths so Colab's default CUDA paths are preserved
current_path = os.environ.get('LD_LIBRARY_PATH', '')
os.environ['LD_LIBRARY_PATH'] = f'/usr/lib64-nvidia:/usr/lib/x86_64-linux-gnu:{current_path}'

# 2. Start Ollama in the background
print("🚀 Starting Ollama server with GPU support...")
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Give the server a few seconds to initialize
time.sleep(5)

print("✅ Setup Phase 2 Complete: Server running!")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

🚀 Starting Ollama server with GPU support...
✅ Setup Phase 2 Complete: Server running!
Tesla T4, 15360 MiB


In [ ]:
from huggingface_hub import hf_hub_download
import subprocess

MODEL_FILE = "Qwen3.5-4B-Uncensored-HauhauCS-Aggressive-Q4_K_M.gguf"

print("📥 Downloading model (this takes a minute)...")
model_path = hf_hub_download(
    repo_id="HauhauCS/Qwen3.5-4B-Uncensored-HauhauCS-Aggressive",
    filename=MODEL_FILE,
    local_dir="/content/ollama-model"
)

# Define the correct Modelfile
modelfile_content = f"FROM {model_path}\n" + """
PARAMETER num_ctx 4096
PARAMETER num_gpu 99
PARAMETER temperature 0.7

TEMPLATE \"\"\"{{ if .System }}
<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}
{{ if .Prompt }}
<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}
<|im_start|>assistant
\"\"\"
"""

# Write and build
with open("/content/Modelfile", "w") as f:
    f.write(modelfile_content)

print("🏗️ Building the model in Ollama...")
subprocess.run(["ollama", "create", "qwen3.5-4b", "-f", "/content/Modelfile"])
print("✅ Setup Phase 3 Complete: Model is ready for use!")

📥 Downloading model (this takes a minute)...


🏗️ Building the model in Ollama...
✅ Setup Phase 3 Complete: Model is ready for use!


In [ ]:
import os
import time
import subprocess
import requests
import json
import gradio as gr

# 1. Kill any broken CPU-bound background processes
subprocess.run(['pkill', '-f', 'ollama'])
time.sleep(3) # Slightly longer sleep to ensure the GPU lock is fully released in Colab

# 2. THE FIX: Prepend to preserve the Colab GPU drivers
env = os.environ.copy()
current_path = env.get('LD_LIBRARY_PATH', '')
env['LD_LIBRARY_PATH'] = f'/usr/lib64-nvidia:/usr/lib/x86_64-linux-gnu:{current_path}'
env['OLLAMA_HOST'] = '127.0.0.1:11434'

# 3. Start the Ollama server in the background WITH the GPU environment
print("🚀 Booting up the Ollama server (GPU linked)...")
subprocess.Popen(['ollama', 'serve'], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# 4. Define the Chat Function (OPTIMIZED)
def chat_with_qwen(message, history):
    # FIX 1: Limit history to the last 3 turns to prevent lag and context crashing
    MAX_HISTORY = 3
    recent_history = history[-MAX_HISTORY:] if len(history) > MAX_HISTORY else history

    messages = []
    for user_msg, bot_msg in recent_history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})
    messages.append({"role": "user", "content": message})

    partial_message = ""
    try:
        # FIX 2: Add connection timeouts so it doesn't hang infinitely
        response = requests.post(
            "http://127.0.0.1:11434/api/chat",
            json={
                "model": "qwen3.5-4b",
                "messages": messages,
                "stream": True
            },
            stream=True,
            timeout=(10, 300) # 10s connect, 300s read timeout
        )
        response.raise_for_status()

        for line in response.iter_lines():
            if line:
                try:
                    # FIX 3: Safely parse chunks so a network hiccup doesn't crash the UI
                    chunk = json.loads(line)
                    if 'message' in chunk:
                        content = chunk['message'].get('content', '')
                        partial_message += content
                        yield partial_message
                except json.JSONDecodeError:
                    continue # Just skip broken packets and keep going

    except Exception as e:
        yield partial_message + f"\n\n[⚠️ Network/Colab Error: The connection dropped, but the server is still alive. ({str(e)})]"
# 5. Build the UI
print("🎨 Building Chat Interface...")
demo = gr.ChatInterface(
    fn=chat_with_qwen,
    title="🚀 Qwen 3.5 Uncensored (GPU Accelerated)",
    description="Running instantly on T4 GPU via Ollama.",
    theme=gr.themes.Soft(primary_hue="teal", neutral_hue="slate")
)

# 6. Launch the UI in the background
print("✅ Server is live! Click the public Gradio link below to chat:")
demo.launch(share=True)

# 7. FORCE THE CELL TO RUN INFINITELY
print("⚙️ Entering infinite loop to keep the cell spinning. Click the stop button to end.")
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Stopped manually. Shutting down gracefully...")
    subprocess.run(['pkill', '-f', 'ollama'])

🚀 Booting up the Ollama server (GPU linked)...
🎨 Building Chat Interface...


/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


✅ Server is live! Click the public Gradio link below to chat:
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7286e4d7f9c653ca4a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


⚙️ Entering infinite loop to keep the cell spinning. Click the stop button to end.


In [ ]:
import time
import requests

print("Sending direct request to Ollama...")
start_time = time.time()

try:
    response = requests.post(
        "http://127.0.0.1:11434/api/generate",
        json={
            "model": "qwen3.5-4b",
            "prompt": "Write a one paragraph story about a fast spaceship.",
            "stream": False
        }
    )

    elapsed_time = time.time() - start_time
    print(f"⏱️ Time taken: {elapsed_time:.2f} seconds\n")
    print(response.json().get('response', 'No response text.'))

except Exception as e:
    print(f"Error: {e}")

Sending direct request to Ollama...
Error: HTTPConnectionPool(host='127.0.0.1', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x794aa320b170>: Failed to establish a new connection: [Errno 111] Connection refused'))
